In [1]:
!pip install -q -U transformers accelerate
!pip install -q peft trl bitsandbytes datasets scikit-learn
print("✅ Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 142.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.3 MB/s eta 0:00:00
✅ Done!


In [2]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted!")

Mounted at /content/drive
✅ Drive mounted!


In [3]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split

filepath = "/content/drive/MyDrive/Colab Notebooks/output.jsonl"

data = []
with open(filepath, "r") as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError:
            continue

df = pd.DataFrame(data)
print(f"Total loaded: {len(df)}")
print(df['label'].value_counts())

# Sample 5000: 70% benign, 30% phishing
benign_df = df[df['label'] == 'benign'].sample(n=3500, random_state=42)
phishing_df = df[df['label'] == 'phishing'].sample(n=1500, random_state=42)
subset = pd.concat([benign_df, phishing_df]).sample(frac=1, random_state=42).reset_index(drop=True)

# 80% train, 20% test
train_df, test_df = train_test_split(subset, test_size=0.2, stratify=subset['label'], random_state=42)

print(f"\nSubset: {len(subset)}")
print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(f"\nTrain labels:\n{train_df['label'].value_counts()}")
print(f"\nTest labels:\n{test_df['label'].value_counts()}")

Total loaded: 123380
label
phishing    77600
benign      45780
Name: count, dtype: int64

Subset: 5000
Train: 4000 | Test: 1000

Train labels:
label
benign      2800
phishing    1200
Name: count, dtype: int64

Test labels:
label
benign      700
phishing    300
Name: count, dtype: int64


In [4]:
import torch
import gc
import time
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# ---------- PROMPT ----------
SYSTEM_PROMPT = "You are a cybersecurity analyst. Analyze the website data and respond with exactly one word: phishing or benign."

def website_to_prompt(row):
    """Turn a row into a prompt for the model."""
    # Get detection signals from families column
    families_str = ""
    if pd.notna(row.get('families')) and isinstance(row['families'], dict):
        signals = []
        for category, items in row['families'].items():
            if isinstance(items, list):
                for item in items[:3]:
                    if isinstance(item, dict):
                        signals.append(f"{item.get('id', '?')} ({item.get('direction', '?')})")
        if signals:
            families_str = "\nDetected signals: " + "; ".join(signals[:8])

    text = row.get('visible_text_excerpt', '') or ''
    if len(text) > 300:
        text = text[:300] + "..."

    return f"""URL: {row.get('url', 'N/A')}
Hostname: {row.get('hostname', 'N/A')}
Registered domain: {row.get('registered_domain', 'N/A')}
Page title: {row.get('title', 'N/A') or 'N/A'}
Visible text: {text}{families_str}

Is this website phishing or benign?"""


# ---------- LOAD MODEL ----------
def load_model(model_name):
    """Load a model in 4-bit quantization."""
    print(f"\n{'='*60}")
    print(f"Loading: {model_name}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name, quantization_config=bnb_config,
        device_map="auto", trust_remote_code=True
    )
    print(f"✅ Loaded! Params: {model.num_parameters():,}")
    return model, tokenizer


# ---------- ASK MODEL ----------
def ask(model, tokenizer, user_prompt):
    """Get a one-word response from the model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=20, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.1,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


# ---------- PARSE RESPONSE ----------
def parse_label(response):
    """Extract phishing or benign from response."""
    r = response.lower()
    if 'phishing' in r:
        return 'phishing'
    if 'benign' in r or 'legitimate' in r or 'safe' in r:
        return 'benign'
    return None


# ---------- EVALUATE ----------
ALL_RESULTS = {}

def evaluate(model, tokenizer, name, eval_df, n=200):
    """Test model on n samples and print metrics."""
    print(f"\n📊 Evaluating: {name} ({n} samples)...")
    sample = eval_df.sample(n=min(n, len(eval_df)), random_state=42)

    y_true, y_pred, failures = [], [], 0
    for i, (_, row) in enumerate(sample.iterrows()):
        if (i + 1) % 50 == 1:
            print(f"  {i+1}/{len(sample)}...")

        resp = ask(model, tokenizer, website_to_prompt(row))
        pred = parse_label(resp)
        if pred is None:
            failures += 1
            pred = 'phishing'

        y_true.append(row['label'])
        y_pred.append(pred)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label='phishing', zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label='phishing', zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label='phishing', zero_division=0)

    print(f"  Acc: {acc:.3f} | Prec: {prec:.3f} | Rec: {rec:.3f} | F1: {f1:.3f} | Failures: {failures}")
    ALL_RESULTS[name] = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
                         'y_true': y_true, 'y_pred': y_pred}
    return acc, f1


# ---------- CLEANUP ----------
def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    print("🧹 GPU cleared.")

print("✅ All functions ready!")

✅ All functions ready!


In [5]:
model1, tok1 = load_model("mistralai/Mistral-7B-Instruct-v0.3")

acc1_base, f1_1_base = evaluate(model1, tok1, "Mistral-7B (baseline)", test_df, 200)

# Sample outputs
print("\nSample responses:")
for lbl in ['phishing', 'benign']:
    row = test_df[test_df['label'] == lbl].iloc[0]
    print(f"  [{lbl.upper()}] → '{ask(model1, tok1, website_to_prompt(row))}'")


Loading: mistralai/Mistral-7B-Instruct-v0.3


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Loaded! Params: 7,248,023,552

📊 Evaluating: Mistral-7B (baseline) (200 samples)...
  1/200...
  51/200...
  101/200...
  151/200...
  Acc: 0.375 | Prec: 0.203 | Rec: 0.481 | F1: 0.286 | Failures: 0

Sample responses:
  [PHISHING] → 'Benign'
  [BENIGN] → 'Phishing'


In [6]:
# Prepare training data
train_dataset = Dataset.from_list([
    {"text": tok1.apply_chat_template([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": website_to_prompt(row)},
        {"role": "assistant", "content": row['label']},
    ], tokenize=False)}
    for _, row in train_df.iterrows()
])
split = train_dataset.train_test_split(test_size=0.1, seed=42)

# Apply LoRA
model1 = prepare_model_for_kbit_training(model1)
model1 = get_peft_model(model1, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none", task_type="CAUSAL_LM",
))
print(f"Trainable params: {model1.get_nb_trainable_parameters()[0]:,}")

# Train
trainer = SFTTrainer(
    model=model1,
    args=SFTConfig(
        output_dir="./mistral-7b-ft",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        warmup_steps=50,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        bf16=True,
        dataset_text_field="text",
        optim="adamw_8bit",
    ),
    train_dataset=split["train"],
    eval_dataset=split["test"],
    processing_class=tok1,
)

start = time.time()
trainer.train()
print(f"✅ Training done in {(time.time()-start)/60:.1f} min")

# Evaluate
acc1_ft, f1_1_ft = evaluate(model1, tok1, "Mistral-7B (fine-tuned)", test_df, 200)

print("\nFine-tuned responses:")
for lbl in ['phishing', 'benign']:
    row = test_df[test_df['label'] == lbl].iloc[0]
    print(f"  [{lbl.upper()}] → '{ask(model1, tok1, website_to_prompt(row))}'")

# Free memory for next model
del model1, tok1, trainer, train_dataset, split
cleanup()

Trainable params: 41,943,040


Adding EOS to train dataset:   0%|          | 0/3600 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3600 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3600 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,1.087132,0.944374
2,0.845393,0.907603
3,0.888646,0.898854


✅ Training done in 37.2 min

📊 Evaluating: Mistral-7B (fine-tuned) (200 samples)...
  1/200...
  51/200...
  101/200...
  151/200...
  Acc: 0.715 | Prec: 0.453 | Rec: 0.462 | F1: 0.457 | Failures: 0

Fine-tuned responses:
  [PHISHING] → 'phishing'
  [BENIGN] → 'benign'
🧹 GPU cleared.


In [ ]:
# TODO: Add your Hugging Face login here before running this notebook.
# from huggingface_hub import login
# login(token="YOUR_HUGGINGFACE_TOKEN_HERE")

In [9]:
model2, tok2 = load_model("meta-llama/Llama-3.2-3B-Instruct")

acc2_base, f1_2_base = evaluate(model2, tok2, "Llama-3.2-3B (baseline)", test_df, 200)

print("\nSample responses:")
for lbl in ['phishing', 'benign']:
    row = test_df[test_df['label'] == lbl].iloc[0]
    print(f"  [{lbl.upper()}] → '{ask(model2, tok2, website_to_prompt(row))}'")


Loading: meta-llama/Llama-3.2-3B-Instruct


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Loaded! Params: 3,212,749,824

📊 Evaluating: Llama-3.2-3B (baseline) (200 samples)...
  1/200...
  51/200...
  101/200...
  151/200...
  Acc: 0.465 | Prec: 0.233 | Rec: 0.462 | F1: 0.310 | Failures: 0

Sample responses:
  [PHISHING] → 'benign'
  [BENIGN] → 'phishing'


In [10]:
# Prepare training data
train_dataset = Dataset.from_list([
    {"text": tok2.apply_chat_template([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": website_to_prompt(row)},
        {"role": "assistant", "content": row['label']},
    ], tokenize=False)}
    for _, row in train_df.iterrows()
])
split = train_dataset.train_test_split(test_size=0.1, seed=42)

# Apply LoRA
model2 = prepare_model_for_kbit_training(model2)
model2 = get_peft_model(model2, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none", task_type="CAUSAL_LM",
))
print(f"Trainable params: {model2.get_nb_trainable_parameters()[0]:,}")

# Train
trainer = SFTTrainer(
    model=model2,
    args=SFTConfig(
        output_dir="./llama-3b-ft",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        warmup_steps=50,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        bf16=True,
        dataset_text_field="text",
        optim="adamw_8bit",
    ),
    train_dataset=split["train"],
    eval_dataset=split["test"],
    processing_class=tok2,
)

start = time.time()
trainer.train()
print(f"✅ Training done in {(time.time()-start)/60:.1f} min")

# Evaluate
acc2_ft, f1_2_ft = evaluate(model2, tok2, "Llama-3.2-3B (fine-tuned)", test_df, 200)

print("\nFine-tuned responses:")
for lbl in ['phishing', 'benign']:
    row = test_df[test_df['label'] == lbl].iloc[0]
    print(f"  [{lbl.upper()}] → '{ask(model2, tok2, website_to_prompt(row))}'")

del model2, tok2, trainer, train_dataset, split
cleanup()

Trainable params: 24,313,856


Adding EOS to train dataset:   0%|          | 0/3600 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3600 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3600 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Epoch,Training Loss,Validation Loss
1,1.389269,1.236000
2,1.168625,1.182875
3,1.179888,1.167303


✅ Training done in 30.0 min

📊 Evaluating: Llama-3.2-3B (fine-tuned) (200 samples)...
  1/200...
  51/200...
  101/200...
  151/200...
  Acc: 0.885 | Prec: 0.723 | Rec: 0.904 | F1: 0.803 | Failures: 0

Fine-tuned responses:
  [PHISHING] → 'phishing'
  [BENIGN] → 'benign'
🧹 GPU cleared.


In [11]:
print("=" * 85)
print("📊 PHISHING WEBSITE DETECTION — MODEL COMPARISON")
print("=" * 85)
print(f"Data: 5000 sites (70% benign, 30% phishing) | Train: {len(train_df)} | Test eval: 200")
print(f"Method: LoRA r=16 α=32 | lr=2e-5 | 3 epochs | adamw_8bit")
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")

print(f"{'Model':<16} │ {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} │ {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} │ {'ΔF1':>6}")
print(f"{'':16} │ {'── Baseline ──':^27} │ {'── Fine-tuned ──':^27} │")
print("─" * 85)

for name in ["Mistral-7B", "Llama-3.2-3B"]:
    bk, fk = f"{name} (baseline)", f"{name} (fine-tuned)"
    if bk in ALL_RESULTS and fk in ALL_RESULTS:
        b, f = ALL_RESULTS[bk], ALL_RESULTS[fk]
        d = f['f1'] - b['f1']
        print(f"{name:<16} │ {b['accuracy']:.3f} {b['precision']:.3f} {b['recall']:.3f} {b['f1']:.3f} │ {f['accuracy']:.3f} {f['precision']:.3f} {f['recall']:.3f} {f['f1']:.3f} │ {d:+.3f}")

print("─" * 85)

print("\n📋 DETAILED REPORTS:")
for key in ALL_RESULTS:
    r = ALL_RESULTS[key]
    print(f"\n--- {key} ---")
    print(classification_report(r['y_true'], r['y_pred'],
          target_names=['Benign', 'Phishing'], digits=3, zero_division=0))

📊 PHISHING WEBSITE DETECTION — MODEL COMPARISON
Data: 5000 sites (70% benign, 30% phishing) | Train: 4000 | Test eval: 200
Method: LoRA r=16 α=32 | lr=2e-5 | 3 epochs | adamw_8bit
Date: 2026-03-18 11:36

Model            │    Acc   Prec    Rec     F1 │    Acc   Prec    Rec     F1 │    ΔF1
                 │       ── Baseline ──        │      ── Fine-tuned ──       │
─────────────────────────────────────────────────────────────────────────────────────
Mistral-7B       │ 0.375 0.203 0.481 0.286 │ 0.715 0.453 0.462 0.457 │ +0.171
Llama-3.2-3B     │ 0.465 0.233 0.462 0.310 │ 0.885 0.723 0.904 0.803 │ +0.494
─────────────────────────────────────────────────────────────────────────────────────

📋 DETAILED REPORTS:

--- Mistral-7B (baseline) ---
              precision    recall  f1-score   support

      Benign      0.649     0.338     0.444       148
    Phishing      0.203     0.481     0.286        52

    accuracy                          0.375       200
   macro avg      0.426     0.409